# VascuMap Batch Pipeline

Runs the full VascuMap pipeline on every image in a source folder.  
- **LIF files**: iterates through each image within the file  
- **TIF/TIFF files**: one pipeline run per file  

Set `record_timings = True` to save a per-image timing breakdown CSV alongside the outputs.

**Default outputs** (always saved):
| File | Description |
|------|-------------|
| `*_overlay_geometry_0.tif` | 2-D device overlay with inner/outer geometry |
| `*_skeleton_overview.png` | Skeleton overview plot |
| `*_analysis_metrics.csv` | Quantitative vascular network metrics |

**Extra outputs** (when `save_all_interim = True`):
| File | Description |
|------|-------------|
| `*_cropped_stack_aligned.npy` | Brightfield stack at 2 µm iso |
| `*_vessel_translation_aligned.npy` | Pix2Pix image translation |
| `*_clean_segmentation.npy` | Cleaned binary vessel segmentation |
| `*_skeleton.npy` | Skeleton from pruned graph |
| `*_holes.npy` | Binary pore mask |
| `*_hole_labels_per_slice.npy` | Per-slice labelled pores |
| `*_hole_distance_per_slice_um.npy` | Inscribed-radius distance map |
| `*_full_graph_skeleton.npy` | Pre-pruning skeleton |
| `*_vessel_mask.npy` | Raw vessel mask |
| `*_graph_nodes.npz` | Sprout/junction node coords |
| `*_clean_graph.pkl` | NetworkX graph (pickle) |

In [1]:
import sys, time
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

import pandas as pd
from liffile import LifFile

from vascumap import VascuMap
from models import Pix2Pix, load_segmentation_model

W0406 22:37:33.205000 586220 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ── Helper functions ───────────────────────────────────────────────────────────

def discover_jobs(source_dir, force_mask=None, require_merged=True):
    """Find .lif/.tif/.tiff files and return (source, files, jobs) list."""
    source = Path(source_dir)
    if not source.is_dir():
        raise FileNotFoundError(f"source_dir does not exist: {source}")

    image_files = sorted(
        p for p in source.iterdir()
        if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
    )

    def _mask_mode(p, img_name=""):
        if force_mask is not None:
            return force_mask
        name = (p.name + " " + img_name).lower()
        if "marina" in name and "bead" in name:
            return "light"
        return "dark" if "marina" in name else False

    jobs = []
    for p in image_files:
        if p.suffix.lower() == ".lif":
            try:
                with LifFile(p) as lif:
                    for idx, img in enumerate(lif.images):
                        name = getattr(img, "name", "")
                        if require_merged and "merged" not in name.lower():
                            continue
                        jobs.append((p, idx, _mask_mode(p, name), name))
            except Exception as exc:
                print(f"[SKIP] {p.name}: {exc}")
        else:
            if require_merged and "merged" not in p.name.lower():
                continue
            jobs.append((p, 0, _mask_mode(p), p.stem))

    return source, image_files, jobs


def get_perfect_image_names(perfect_dir):
    """Return a set of image names that have already been perfectly segmented.

    Reads subfolder names from the CATEGORISED/Perfect directory.
    """
    perfect = Path(perfect_dir)
    if not perfect.is_dir():
        print(f"[WARN] Perfect directory not found: {perfect}")
        return set()
    names = {d.name for d in perfect.iterdir() if d.is_dir()}
    print(f"Found {len(names)} perfect images in {perfect}")
    return names


def run_batch(jobs, output_base, device_width_um, brightfield_channel=0,
              save_all_interim=False, model_p2p=None, model_unet=None,
              record_timings=False, start_index=1, skip_names=None):
    """Run all jobs and optionally save a timing CSV.

    Parameters
    ----------
    skip_names : set or None
        Image names to skip (e.g. already-perfect segmentations).
    """
    results, timing_rows = [], []
    if skip_names is None:
        skip_names = set()

    for i, (path, idx, mask_flag, img_name) in enumerate(jobs, start_index):
        tag = f" (LIF #{idx}: {img_name})" if path.suffix.lower() == ".lif" else ""

        # Build the image name the same way the loop body does, so we can
        # check it against the skip list before doing any work.
        if path.suffix.lower() == ".lif":
            safe_name = img_name.replace("/", "_").replace("\\", "_")
            expected_name = f"{path.stem}_{safe_name}_img{idx}"
        else:
            expected_name = path.stem

        if expected_name in skip_names:
            print(f"\n[{i}/{len(jobs)}] {path.name}{tag}  — SKIP (already perfect)")
            results.append((expected_name, "SKIP_PERFECT", ""))
            continue

        print(f"\n{'='*70}\n[{i}/{len(jobs)}] {path.name}{tag}  mask={mask_flag}\n{'='*70}")

        try:
            t0 = time.time()
            vm = VascuMap(
                use_device_segmentation_app=False,
                image_source_path=str(path),
                image_index=idx,
                device_width_um=device_width_um,
                mask_central_region=mask_flag,
                brightfield_channel=brightfield_channel,
                model_p2p=model_p2p,
                model_unet=model_unet,
            )
            vm.image_name = expected_name
            out_dir = Path(output_base) / vm.image_name
            print(f"  Output → {out_dir}")
            vm.pipeline(output_dir=out_dir, save_all_interim=save_all_interim)
            wall = time.time() - t0
            results.append((vm.image_name, "OK", ""))
            if record_timings:
                timing_rows.append({
                    "image_name": vm.image_name, "source_file": path.name,
                    "lif_image_name": img_name,
                    "image_index": idx, "status": "OK",
                    "t_device_seg_s": round(getattr(vm, "_t_device_seg", 0), 1),
                    "t_preprocess_s": round(getattr(vm, "_t_preprocess", 0), 1),
                    "t_inference_s":  round(getattr(vm, "_t_inference", 0), 1),
                    "t_analysis_s":   round(getattr(vm, "_t_analysis", 0), 1),
                    "t_pipeline_s":   round(getattr(vm, "_t_total", 0), 1),
                    "t_job_wall_s":   round(wall, 1),
                })
        except Exception as exc:
            print(f"  [SKIP] {exc}")
            results.append((path.name + tag, "FAILED", str(exc)))
            if record_timings:
                timing_rows.append({
                    "image_name": path.name + tag, "source_file": path.name,
                    "lif_image_name": img_name,
                    "image_index": idx, "status": "FAILED",
                })

        if record_timings and timing_rows:
            pd.DataFrame(timing_rows).to_csv(Path(output_base) / "batch_timings.csv", index=False)

    # Summary
    n_ok = sum(1 for _, s, _ in results if s == "OK")
    n_skip = sum(1 for _, s, _ in results if s == "SKIP_PERFECT")
    print(f"\n{'='*70}\nBatch complete: {n_ok}/{len(results)} succeeded, {n_skip} skipped (perfect)")
    for name, status, msg in results:
        if status == "FAILED":
            print(f"  FAILED: {name}: {msg}")
    if record_timings and timing_rows:
        print(f"\nTimings → {Path(output_base) / 'batch_timings.csv'}")
        cols = ["image_name", "lif_image_name", "t_device_seg_s", "t_preprocess_s", "t_inference_s", "t_analysis_s", "t_pipeline_s"]
        print(pd.DataFrame(timing_rows).reindex(columns=cols).to_string(index=False))

    return results

In [3]:
shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)


In [4]:
# ── Configuration (edit here) ──────────────────────────────────────────────────
source_dir  = r"Z:\Bel\Marina_Vascumap\Inputs"
output_base = r"Z:\Bel\Marina_Vascumap\Outputs"

device_width_um       = 35.0
brightfield_channel   = 0
force_mask            = None     # "dark" / "light" / False / None (auto)
require_merged        = False
save_all_interim      = False
record_timings        = True

In [ ]:
# ── Discover jobs + run ────────────────────────────────────────────────────────
source, image_files, jobs = discover_jobs(source_dir, force_mask, require_merged)

print(f"Source: {source}\nFiles: {len(image_files)}  |  Jobs: {len(jobs)}")
for i, (p, idx, mask, img_name) in enumerate(jobs, 1):
    tag = f" (LIF image {idx}: '{img_name}')" if p.suffix.lower() == ".lif" else ""
    print(f"  {i}. {p.name}{tag}  mask={mask}")

# Skip images already categorised as perfect
perfect_dir = r"Z:\Bel\Marina_Vascumap\Outputs\CATEGORISED\Perfect"
perfect_names = get_perfect_image_names(perfect_dir)

run_batch(jobs, output_base, device_width_um, brightfield_channel,
          save_all_interim, shared_model_p2p, shared_model_unet, record_timings,
          skip_names=perfect_names)

Source: Z:\Bel\Marina_Vascumap\Inputs
Files: 8  |  Jobs: 214
  1. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 0: 'Bead_dev1_Merged')  mask=light
  2. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 1: 'Bead_dev2_Merged')  mask=light
  3. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 2: 'Bead_dev3_Merged')  mask=light
  4. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 3: 'Bead_dev4_Merged')  mask=light
  5. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 4: 'Bead_dev5_Merged')  mask=light
  6. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 5: 'Bead_dev6_Merged')  mask=light
  7. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 6: 'rLN3_dev1_Merged')  mask=dark
  8. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 7: 'rLN3_dev2_Merged')  mask=dark
  9. marina_2024.11.14_MCL1_MCL3_M4_morphologyBFseeding1.lif (LIF image 8: 'rLN3_dev3

Processing chunks: 100%|██████████| 3/3 [00:33<00:00, 11.27s/it]


strong contiguous vote planes 0-12
  ⏱  Stage 2 (Pix2Pix + UNet): 1240.3s
  Trimmed 21 top / 0 bottom over-segmented z-slices
Running skeletonisation and analysis...
Organoid exclusion mask applied: 846349 pixels excluded per z-slice


Processing chunks: 100%|██████████| 1/1 [00:04<00:00,  4.06s/it]


 chip_volume_um3  convex_hull_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_sprout_and_branch_length_um  p90_minus_p10_sprout_and_branch_length_um  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3
    2524772800.0            2756318224.0       1389831352.0                0.504235           611845.4124

...

Processing chunks: 100%|██████████| 3/3 [00:32<00:00, 10.84s/it]


strong contiguous vote planes 0-8
  ⏱  Stage 2 (Pix2Pix + UNet): 1280.5s
Running skeletonisation and analysis...
Organoid exclusion mask applied: 580251 pixels excluded per z-slice


Processing chunks: 100%|██████████| 1/1 [00:04<00:00,  4.55s/it]


 chip_volume_um3  convex_hull_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_sprout_and_branch_length_um  p90_minus_p10_sprout_and_branch_length_um  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3
    2590102440.0            2.720842e+09       1531006184.0                0.562696           630987.5058

100%|██████████| 112/112 [00:24<00:00,  4.54it/s]
